# 04 · Đánh giá tổng hợp + đẩy model lên Hugging Face Hub

Input: code + output của 01 (splits, baseline, nrl), 02 (export ONNX), 03 (runs/s2s).
Bật GPU để seq2seq decode nhanh; tagger ONNX đo trên CPU 1 luồng cho giống máy yếu.
Secrets cần có: `HF_TOKEN` (write). Đổi `HF_REPO` thành tên của bạn trên HF.

In [ ]:
import glob, json, os, sys

CODE = glob.glob("/kaggle/input/**/vi_diacritics/__init__.py", recursive=True)[0].rsplit("/vi_diacritics", 1)[0]
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def find(pat):
    return glob.glob(f"/kaggle/input/**/{pat}", recursive=True)[0]


SPLITS = os.path.dirname(find("splits/train.txt"))
EVAL = os.path.dirname(find("eval/nrl_all.txt"))
EXPORT = os.path.dirname(find("export/tagger.onnx"))
S2S = os.path.dirname(find("s2s/best.pt"))
BASELINE = find("baseline.json")
REAL = find("real_typing.txt")
HF_REPO = "DuongCodeAI/vi-diacritics-tagger"
!pip install -q onnxruntime tokenizers

In [ ]:
tests = [f"wiki={SPLITS}/test.txt", f"real={REAL}"] + [
    f"{os.path.basename(p)[:-4]}={p}" for p in sorted(glob.glob(f"{EVAL}/nrl_*.txt"))]
args = " ".join(f"--test {t}" for t in tests)
!python -m vi_diacritics.evaluate --export {EXPORT} --baseline {BASELINE} --s2s-run {S2S} {args} --limit 3000 --out results

In [ ]:
print(open("results/eval.md", encoding="utf-8").read())

## Phân tích lỗi: âm tiết nào khó nhất, nhầm thành gì

In [ ]:
import pandas as pd

err = json.load(open("results/errors_wiki_tagger_int8.json", encoding="utf-8"))
display(pd.DataFrame(err["hard_syllables"], columns=["âm tiết", "lỗi", "tổng", "tỉ lệ lỗi"]).head(20))
display(pd.DataFrame(err["confusions"], columns=["đúng", "đoán", "số lần"]).head(20))

In [ ]:
err = json.load(open("results/errors_real_tagger_int8.json", encoding="utf-8"))
for e in err["examples"][:20]:
    print("IN  :", e["input"])
    print("PRED:", e["pred"])
    print("GOLD:", e["gold"])
    print()

## Đẩy ONNX lên HF Hub (để repo khác + web demo tải về)

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=EXPORT, repo_id=HF_REPO)
api.upload_folder(folder_path="results", repo_id=HF_REPO, path_in_repo="results")
print(f"https://huggingface.co/{HF_REPO}")